# Data Preprocessing

This notebook constructs the model-ready training, validation, and human test datasets from the original PolitiSky24 data. It recovers timestamps for the retrieved context posts, handles ambiguous post matches, and creates user-disjoint dataset splits.

## 1. Prepare the stance-detection post IDs

The `user_post_list_for_stance_detection.parquet` dataset contains one row per user, where `PostIdList` stores the IDs of the posts that were available for stance detection.

In this step, we transform these lists into a flat `(UserId, PostId)` table. This allows us to efficiently select only the relevant posts from the much larger posting-history dataset in the next step.

The resulting table has the structure:

| UserId | PostId |
|---|---|
| 9 | 12167384 |
| 9 | 4538012 |
| 9 | 7809 |

In [9]:
import os
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc


raw_path = "../data/raw"
preprocessed_path = "../data/preprocessed"

os.makedirs(preprocessed_path, exist_ok=True)


# Load the post lists used for stance detection
post_lists = pq.read_table(
    f"{raw_path}/user_post_list_for_stance_detection.parquet"
)

print(f"Users: {post_lists.num_rows:,}")
display(post_lists.slice(0, 5).to_pandas())
print(post_lists.schema)


# Flatten the PostId lists
post_ids = pc.list_flatten(
    post_lists["PostIdList"].combine_chunks()
)

parent_indices = pc.list_parent_indices(
    post_lists["PostIdList"].combine_chunks()
)

user_ids = pc.take(
    post_lists["UserId"].combine_chunks(),
    parent_indices
)

stance_detection_posts = pa.table({
    "UserId": user_ids,
    "PostId": post_ids
})

display(stance_detection_posts.slice(0, 10).to_pandas())

Users: 8,467


,UserId,PostIdList
0,9,"[12167384, 4538012, 7809, 3924058, 7807, 48816..."
1,10,"[2620214, 12666, 12665, 12674, 12248774, 16645..."
2,49,"[13503, 13501, 13496, 13495, 13493, 13491, 134..."
3,105,"[14930, 14927, 14924, 14921, 369225, 14979, 14..."
4,144,"[15305, 15303, 15301, 15300, 15298, 15297, 152..."


UserId: int64
PostIdList: list<element: int64>
  child 0, element: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 495


,UserId,PostId
0,9,12167384
1,9,4538012
2,9,7809
3,9,3924058
4,9,7807
5,9,4881667
6,9,7805
7,9,7793
8,9,5778
9,9,5775


In [10]:
# Sanity checks
duplicate_pairs = (
    stance_detection_posts
    .to_pandas()
    .duplicated(["UserId", "PostId"])
    .sum()
)

print(f"User-post pairs: {stance_detection_posts.num_rows:,}")
print(
    f"Unique users: "
    f"{pc.count_distinct(stance_detection_posts['UserId']).as_py():,}"
)
print(
    f"Unique posts: "
    f"{pc.count_distinct(stance_detection_posts['PostId']).as_py():,}"
)
print(f"Duplicate user-post pairs: {duplicate_pairs:,}")


# Save for filtering the full posting history
stance_detection_post_ids_path = (
    f"{preprocessed_path}/stance_detection_post_ids.parquet"
)

pq.write_table(
    stance_detection_posts,
    stance_detection_post_ids_path,
    compression="snappy"
)

print(f"\nSaved to: {stance_detection_post_ids_path}")

User-post pairs: 2,845,217
Unique users: 8,467
Unique posts: 1,768,725
Duplicate user-post pairs: 0

Saved to: ../data/preprocessed/stance_detection_post_ids.parquet


---
## 2. Extract relevant posts from the posting history

The full posting-history dataset contains all available posts and is considerably larger than what we need for the following preprocessing steps.

Using the `(UserId, PostId)` pairs prepared in Section 1, we filter the posting history to the posts that were available for stance detection. Both columns are used for matching, since a `PostId` should not be treated as a standalone identifier across users.

For the matching posts, we retain:

- `UserId`
- `PostId`
- `PostTime`
- `Content`

This filtered dataset will later serve as a lookup table for recovering metadata of the retrieved context posts.

> **Note:** The file `user_post_history_dataset.parquet` is approximately 2.1 GB and therefore cannot be uploaded to GitHub. This code only works if the original `user_post_history_dataset.parquet` is available in `data/raw`. We ran this step locally and uploaded the resulting preprocessed file to `data/preprocessed/filtered_post_history.parquet`.

In [11]:
history_path = f"{raw_path}/user_post_history_dataset.parquet"
filtered_history_path = (
    f"{preprocessed_path}/filtered_post_history.parquet"
)

# Load the relevant UserId-PostId pairs
stance_detection_posts = pq.read_table(
    stance_detection_post_ids_path,
    columns=["UserId", "PostId"]
)

print(f"Relevant user-post pairs: {stance_detection_posts.num_rows:,}")


# Filter the large posting-history file batch by batch
history_file = pq.ParquetFile(history_path)

columns = [
    "UserId",
    "PostId",
    "PostTime",
    "Content"
]

output_schema = pa.schema([
    history_file.schema_arrow.field(column)
    for column in columns
])

with pq.ParquetWriter(
    filtered_history_path,
    output_schema,
    compression="snappy"
) as writer:

    for batch in history_file.iter_batches(
        batch_size=250_000,
        columns=columns
    ):
        batch_table = pa.Table.from_batches([batch])

        # Match using both UserId and PostId
        filtered_batch = batch_table.join(
            stance_detection_posts,
            keys=["UserId", "PostId"],
            join_type="inner"
        )

        if filtered_batch.num_rows > 0:
            writer.write_table(filtered_batch)

print(f"Saved to: {filtered_history_path}")

Relevant user-post pairs: 2,845,217
Saved to: ../data/preprocessed/filtered_post_history.parquet


In [12]:
filtered_history = pq.read_table(filtered_history_path)

print(f"Filtered history rows: {filtered_history.num_rows:,}")
print(
    f"Unique users: "
    f"{pc.count_distinct(filtered_history['UserId']).as_py():,}"
)

print("\nSchema:")
print(filtered_history.schema)

display(filtered_history.slice(0, 5).to_pandas())


# Check that every required UserId-PostId pair was found
filtered_pairs = (
    filtered_history
    .select(["UserId", "PostId"])
    .group_by(["UserId", "PostId"])
    .aggregate([])
)

missing_pairs = stance_detection_posts.join(
    filtered_pairs,
    keys=["UserId", "PostId"],
    join_type="left anti"
)

print(f"Missing user-post pairs: {missing_pairs.num_rows:,}")

assert missing_pairs.num_rows == 0, (
    "Some stance-detection UserId-PostId pairs are missing "
    "from the filtered posting history."
)

Filtered history rows: 2,845,217
Unique users: 8,467

Schema:
UserId: int64
PostId: int64
PostTime: string
Content: string


,UserId,PostId,PostTime,Content
0,213384,9738846,2024-11-19T21:07:40.808Z,Too bad he’ll roll back the drug price negotia...
1,363769,16537228,2024-11-19T21:07:37.376Z,@anonymous Why I followed back: you had me at ...
2,100747,4623479,2024-11-19T21:07:36.021Z,One of my hobbies is photography. Might as wel...
3,319615,14635127,2024-11-19T21:07:35.424Z,"We're off to see the wizard, the wonderful wiz..."
4,84980,3984055,2024-11-19T21:07:33.145Z,@anonymous \n#Recount2024


Missing user-post pairs: 0


---
## 3. Prepare the retrieved LLM context posts

The `llm_annotated_full_user_stance_dataset.parquet` contains one row per user-target pair. Each row contains the stance annotation as well as the `Context` posts that were retrieved and used for the LLM-based stance detection.

Instead of using the user's complete posting history, these retrieved context posts will serve as the input for our stance-classification experiments.

In this step, we:

1. Load the relevant columns from the LLM-annotated dataset.
2. Flatten the `Context` lists so that each retrieved context post temporarily occupies one row.

The resulting context table has the structure:

| UserId | TargetEntity | Content |
|---|---|---|
| 9 | Harris | ... |
| 9 | Harris | ... |
| 9 | Trump | ... |

In the next step, these context posts will be matched against the filtered posting history to recover their `PostId` and `PostTime`.

In [13]:
llm_path = f"{raw_path}/llm_annotated_full_user_stance_dataset.parquet"

llm_data = pq.read_table(
    llm_path,
    columns=[
        "UserId",
        "TargetEntity",
        "StanceLabel",
        "ConfidenceLevel",
        "Context"
    ]
)

print(f"User-target pairs: {llm_data.num_rows:,}")
print(
    f"Unique users: "
    f"{pc.count_distinct(llm_data['UserId']).as_py():,}"
)

print("\nSchema:")
print(llm_data.schema)

display(llm_data.slice(0, 5).to_pandas())


# Each user-target pair should occur exactly once
llm_pairs = llm_data.select(["UserId", "TargetEntity"]).to_pandas()

assert not llm_pairs.duplicated().any(), (
    "Duplicate UserId-TargetEntity pairs found."
)

User-target pairs: 16,044
Unique users: 8,022

Schema:
UserId: int64
TargetEntity: string
StanceLabel: string
ConfidenceLevel: double
Context: list<element: string>
  child 0, element: string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1485


,UserId,TargetEntity,StanceLabel,ConfidenceLevel,Context
0,9,Harris,Favor,0.85,[Assuming Harris is the pick. Biden might have...
1,9,Trump,Against,0.95,[Gonna fucking Battle Royale us into a Trump p...
2,10,Harris,Favor,0.95,"[Kamala/Harris, Kamala Harris is very good at ..."
3,10,Trump,Against,0.95,"[Not to get too political on here, but I reall..."
4,49,Harris,Against,0.85,[To any and ALL MAGA calling for unity and the...


In [14]:
contexts = llm_data["Context"].combine_chunks()

# Check the number of retrieved context posts
context_lengths = pc.list_value_length(contexts)

min_contexts = pc.min(context_lengths).as_py()
max_contexts = pc.max(context_lengths).as_py()

print(
    f"Context posts per user-target pair: "
    f"{min_contexts} - {max_contexts}"
)

assert max_contexts <= 10, (
    "A user-target pair contains more than 10 context posts."
)


# Flatten the context lists
contents = pc.list_flatten(contexts)
parent_indices = pc.list_parent_indices(contexts)

context_posts = pa.table({
    "UserId": pc.take(
        llm_data["UserId"].combine_chunks(),
        parent_indices
    ),
    "TargetEntity": pc.take(
        llm_data["TargetEntity"].combine_chunks(),
        parent_indices
    ),
    "Content": contents
})

assert context_posts["Content"].null_count == 0, (
    "Missing Content found among retrieved context posts."
)

print(f"Context posts: {context_posts.num_rows:,}")

display(context_posts.slice(0, 15).to_pandas())

Context posts per user-target pair: 1 - 10
Context posts: 119,954


,UserId,TargetEntity,Content
0,9,Harris,Assuming Harris is the pick. Biden might have ...
1,9,Harris,"I mean so do I, there's several people I'd wan..."
2,9,Harris,Oh COME ON! Like EVERYONE agrees Bibi is terri...
3,9,Harris,I'm not gonna deny that the idea of Alina Habb...
4,9,Harris,All my doom and gloom aside. If Harris needs a...
5,9,Harris,"I don't see how you read ""We are withholding o..."
6,9,Harris,"Cannot express how badly this whole ""Dump Bide..."
7,9,Harris,Jill Stein is trash and cannot be trusted.\n\n...
8,9,Trump,Gonna fucking Battle Royale us into a Trump pr...
9,9,Trump,"I don't see how you read ""We are withholding o..."


---
## 4. Match context posts to the original posting history

The retrieved context posts contain their textual `Content`, but not the corresponding `PostId` or `PostTime`.

We first match each context post against `filtered_post_history.parquet` using `(UserId, Content)`, since the `PostId` is not yet known at this stage.

Before determining ambiguity, we verify that every retrieved context post has at least one match in the original posting history. A context post without any match would indicate a preprocessing problem and stops the pipeline.

For each successfully matched context post, we then distinguish between:

- **One match:** The original post can be identified uniquely. We retain its `PostId` and recover its `PostTime` using `(UserId, PostId)`. `IsAmbiguous` is set to `False`.
- **Multiple matches:** The same user posted identical content multiple times. We cannot determine which original post was retrieved, so `PostId` and `PostTime` are set to null and `IsAmbiguous` is set to `True`.

A temporary `ContextPostId` keeps individual retrieved context entries separate during matching. It is only needed during preprocessing and will not be part of the final dataset.

In [26]:
history_lookup = pq.read_table(
    filtered_history_path,
    columns=[
        "UserId",
        "PostId",
        "PostTime",
        "Content"
    ]
)

assert history_lookup["PostId"].null_count == 0


# Reset in case the cell is rerun
context_posts = context_posts.select([
    "UserId",
    "TargetEntity",
    "Content"
])

# Temporary identifier for each individual context entry
context_posts = context_posts.append_column(
    "ContextPostId",
    pa.array(
        np.arange(context_posts.num_rows),
        type=pa.int64()
    )
)

# At this point PostId is unknown, so matching uses UserId + Content
context_matches = context_posts.join(
    history_lookup,
    keys=["UserId", "Content"],
    join_type="left outer"
)


# Every context post should have at least one match
num_unmatched = context_matches["PostId"].null_count

print(f"Context posts: {context_posts.num_rows:,}")
print(f"Unmatched context posts: {num_unmatched:,}")

assert num_unmatched == 0, (
    "Some context posts could not be matched to the original posting history."
)

print(f"Rows after matching: {context_matches.num_rows:,}")

display(context_matches.slice(0, 10).to_pandas())

Context posts: 119,954
Unmatched context posts: 0
Rows after matching: 231,768


,UserId,TargetEntity,Content,ContextPostId,PostId,PostTime
0,330551,Harris,This is the rallying call that I've needed to ...,98304,10181351,2024-11-20T14:59:05.639Z
1,330551,Harris,I will forever remain deeply ashamed of having...,98305,1450975,2024-11-11T02:23:06.835Z
2,330551,Harris,My statement on leaving Twitter,98306,17876256,2024-11-13T16:15:30.596Z
3,330551,Harris,When Nikki Haley is mocking you for being a fr...,98307,14421125,2024-11-18T22:02:47.468Z
4,330551,Trump,I heard The Orange Shitgibbon doesn't like it ...,98308,8648016,2024-11-25T21:47:00.198Z
5,330551,Trump,It's almost like Donald Trump is deliberately ...,98309,12163848,2024-11-13T22:25:52.727Z
6,330551,Trump,This is the fucker Trump wants running the DoD.,98310,10995796,2024-11-13T04:04:45.093Z
7,330551,Trump,Seems like sexual assault is a pre-requisite t...,98311,14751634,2024-11-17T01:44:01.187Z
8,330551,Trump,Trump nominating Matt Gaetz is clear proof he ...,98312,14406973,2024-11-13T22:36:18.404Z
9,330551,Trump,Please welcome Mary Trump to Bluesky and sprea...,98313,11674455,2024-11-14T18:24:54.735Z


In [16]:
# Count how many original posts match each context entry
match_summary = (
    context_matches
    .group_by([
        "ContextPostId",
        "UserId"
    ])
    .aggregate([
        ("PostId", "count_distinct"),
        ("PostId", "min")
    ])
)

match_summary = match_summary.rename_columns([
    "ContextPostId",
    "UserId",
    "NumMatches",
    "MatchedPostId"
])


# Add match information back to the context posts
resolved_context_posts = context_posts.join(
    match_summary,
    keys=["ContextPostId", "UserId"],
    join_type="left outer"
)

is_ambiguous = pc.greater(
    resolved_context_posts["NumMatches"],
    1
)

num_ambiguous = pc.sum(
    pc.cast(is_ambiguous, pa.int64())
).as_py()

print(
    f"Ambiguous context posts: {num_ambiguous:,} "
    f"({num_ambiguous / resolved_context_posts.num_rows:.2%})"
)


# Keep PostId only if the original post is uniquely identifiable
resolved_context_posts = resolved_context_posts.append_column(
    "PostId",
    pc.if_else(
        is_ambiguous,
        pa.scalar(
            None,
            type=resolved_context_posts["MatchedPostId"].type
        ),
        resolved_context_posts["MatchedPostId"]
    )
)

resolved_context_posts = resolved_context_posts.append_column(
    "IsAmbiguous",
    is_ambiguous
)


# Recover PostTime using UserId + PostId
post_time_lookup = history_lookup.select([
    "UserId",
    "PostId",
    "PostTime"
])

resolved_context_posts = resolved_context_posts.join(
    post_time_lookup,
    keys=["UserId", "PostId"],
    join_type="left outer"
)


# Matching must preserve every individual context entry
assert resolved_context_posts.num_rows == context_posts.num_rows, (
    "Joining on UserId and PostId changed the number of context posts."
)

resolved_context_posts = resolved_context_posts.select([
    "ContextPostId",
    "UserId",
    "TargetEntity",
    "PostId",
    "Content",
    "PostTime",
    "IsAmbiguous"
])

display(resolved_context_posts.slice(0, 15).to_pandas())

Ambiguous context posts: 1,565 (1.30%)


,ContextPostId,UserId,TargetEntity,PostId,Content,PostTime,IsAmbiguous
0,98304,330551,Harris,10181351,This is the rallying call that I've needed to ...,2024-11-20T14:59:05.639Z,False
1,98305,330551,Harris,1450975,I will forever remain deeply ashamed of having...,2024-11-11T02:23:06.835Z,False
2,98306,330551,Harris,17876256,My statement on leaving Twitter,2024-11-13T16:15:30.596Z,False
3,98307,330551,Harris,14421125,When Nikki Haley is mocking you for being a fr...,2024-11-18T22:02:47.468Z,False
4,98308,330551,Trump,8648016,I heard The Orange Shitgibbon doesn't like it ...,2024-11-25T21:47:00.198Z,False
5,98309,330551,Trump,12163848,It's almost like Donald Trump is deliberately ...,2024-11-13T22:25:52.727Z,False
6,98310,330551,Trump,10995796,This is the fucker Trump wants running the DoD.,2024-11-13T04:04:45.093Z,False
7,98311,330551,Trump,14751634,Seems like sexual assault is a pre-requisite t...,2024-11-17T01:44:01.187Z,False
8,98312,330551,Trump,14406973,Trump nominating Matt Gaetz is clear proof he ...,2024-11-13T22:36:18.404Z,False
9,98313,330551,Trump,11674455,Please welcome Mary Trump to Bluesky and sprea...,2024-11-14T18:24:54.735Z,False


---
## 5. Build the final LLM context table

After matching the retrieved context posts to the original posting history, we now group them back into one row per `(UserId, TargetEntity)` pair.

Each `ContextPosts` entry contains:

- `PostId`
- `Content`
- `PostTime`
- `IsAmbiguous`

For ambiguous posts, `PostId` and `PostTime` remain null while `IsAmbiguous = True`.

We also add two summary columns:

- `NumContextPosts`: Number of retrieved context posts for the user-target pair.
- `NumAmbiguousTimestamps`: Number of context posts whose original post and timestamp could not be identified uniquely.

Finally, the context information is combined with the original `StanceLabel` and `ConfidenceLevel` from the LLM-annotated dataset.

The resulting table has one row per user-target pair.

In [17]:
from collections import defaultdict


# Order context posts from newest to oldest
# Ambiguous posts have no timestamp and are placed last
resolved_context_posts = resolved_context_posts.sort_by([
    ("UserId", "ascending"),
    ("TargetEntity", "ascending"),
    ("PostTime", "descending"),
    ("ContextPostId", "ascending")
])

grouped_context = defaultdict(list)

for row in resolved_context_posts.to_pylist():
    key = (row["UserId"], row["TargetEntity"])

    grouped_context[key].append({
        "PostId": row["PostId"],
        "Content": row["Content"],
        "PostTime": row["PostTime"],
        "IsAmbiguous": row["IsAmbiguous"]
    })


# Build ContextPosts in the same row order as llm_data
context_lists = []

for row in llm_data.select(["UserId", "TargetEntity"]).to_pylist():
    key = (row["UserId"], row["TargetEntity"])
    context_lists.append(grouped_context[key])


context_type = pa.list_(
    pa.struct([
        pa.field("PostId", resolved_context_posts["PostId"].type),
        pa.field("Content", pa.string()),
        pa.field("PostTime", resolved_context_posts["PostTime"].type),
        pa.field("IsAmbiguous", pa.bool_())
    ])
)

num_context_posts = [
    len(posts)
    for posts in context_lists
]

num_ambiguous_timestamps = [
    sum(post["IsAmbiguous"] for post in posts)
    for posts in context_lists
]


llm_context_data = pa.table({
    "UserId": llm_data["UserId"],
    "TargetEntity": llm_data["TargetEntity"],
    "StanceLabel": llm_data["StanceLabel"],
    "ConfidenceLevel": llm_data["ConfidenceLevel"],
    "ContextPosts": pa.array(
        context_lists,
        type=context_type
    ),
    "NumContextPosts": pa.array(
        num_context_posts,
        type=pa.int16()
    ),
    "NumAmbiguousTimestamps": pa.array(
        num_ambiguous_timestamps,
        type=pa.int16()
    )
})

display(llm_context_data.slice(0, 5).to_pandas())

,UserId,TargetEntity,StanceLabel,ConfidenceLevel,ContextPosts,NumContextPosts,NumAmbiguousTimestamps
0,9,Harris,Favor,0.85,"[{'PostId': 7153, 'Content': 'I don't see how ...",8,0
1,9,Trump,Against,0.95,"[{'PostId': 7153, 'Content': 'I don't see how ...",7,0
2,10,Harris,Favor,0.95,"[{'PostId': 11913, 'Content': 'Kamala Harris i...",8,0
3,10,Trump,Against,0.95,"[{'PostId': 12358, 'Content': 'AP still has Tr...",9,0
4,49,Harris,Against,0.85,"[{'PostId': 13503, 'Content': 'Let be real @an...",6,0


In [18]:
for posts in context_lists:
    timestamps = [
        post["PostTime"]
        for post in posts
        if post["PostTime"] is not None
    ]

    assert timestamps == sorted(timestamps, reverse=True), (
        "Context posts are not ordered from newest to oldest."
    )

In [19]:
# Final structural checks
assert llm_context_data.num_rows == llm_data.num_rows

assert sum(num_context_posts) == resolved_context_posts.num_rows, (
    "Some context posts were lost or duplicated during grouping."
)

total_ambiguous = sum(num_ambiguous_timestamps)

assert total_ambiguous == pc.sum(
    pc.cast(
        resolved_context_posts["IsAmbiguous"],
        pa.int64()
    )
).as_py(), (
    "Ambiguous-post counts do not match."
)

assert max(num_context_posts) <= 10


# Row-level ambiguity statistics
rows_with_ambiguity = sum(
    count > 0
    for count in num_ambiguous_timestamps
)

print(f"User-target rows: {llm_context_data.num_rows:,}")
print(f"Total context posts: {sum(num_context_posts):,}")
print(f"Ambiguous context posts: {total_ambiguous:,}")

print(
    f"Rows with at least one ambiguous timestamp: "
    f"{rows_with_ambiguity:,} "
    f"({rows_with_ambiguity / llm_context_data.num_rows:.2%})"
)


# Save the final LLM context table
llm_context_path = (
    f"{preprocessed_path}/llm_user_target_contexts.parquet"
)

pq.write_table(
    llm_context_data,
    llm_context_path,
    compression="snappy"
)

print(f"\nSaved to: {llm_context_path}")

User-target rows: 16,044
Total context posts: 119,954
Ambiguous context posts: 1,565
Rows with at least one ambiguous timestamp: 1,032 (6.43%)

Saved to: ../data/preprocessed/llm_user_target_contexts.parquet


---
## 6. Build the human gold evaluation dataset

The human-annotated validation dataset contains the gold stance labels for the held-out validation users, while `llm_annotated_validation_user_stance_dataset.parquet` contains the retrieved `Context` posts for the same users.

To construct the human evaluation dataset, we:

1. Load the retrieved context posts from the LLM validation dataset.
2. Reshape the human annotations from separate `Trump` and `Harris` columns into one row per `(UserId, TargetEntity)` pair.
3. Process the validation context posts using the same matching and ambiguity logic as for the full LLM dataset.
4. Group the context posts back into one `ContextPosts` list per user-target pair.
5. Attach the human gold `StanceLabel`.

The resulting table therefore uses the same input representation as the LLM training data, but the stance labels come from human annotations rather than the LLM.

In [20]:
llm_validation_path = (
    f"{raw_path}/llm_annotated_validation_user_stance_dataset.parquet"
)
human_labels_path = (
    f"{raw_path}/human_annotated_validation_user_stance_dataset.parquet"
)


# Load validation contexts
llm_validation = pq.read_table(
    llm_validation_path,
    columns=[
        "UserId",
        "TargetEntity",
        "Context"
    ]
)

# Load and reshape human gold labels
human_labels = pq.read_table(
    human_labels_path
).to_pandas()

human_gold = human_labels.melt(
    id_vars="UserId",
    value_vars=["Trump", "Harris"],
    var_name="TargetEntity",
    value_name="StanceLabel"
)

human_gold = pa.Table.from_pandas(
    human_gold,
    preserve_index=False
)


# Both datasets should contain exactly the same user-target pairs
validation_pairs = (
    llm_validation
    .select(["UserId", "TargetEntity"])
)

gold_pairs = (
    human_gold
    .select(["UserId", "TargetEntity"])
)

missing_in_gold = validation_pairs.join(
    gold_pairs,
    keys=["UserId", "TargetEntity"],
    join_type="left anti"
)

missing_in_validation = gold_pairs.join(
    validation_pairs,
    keys=["UserId", "TargetEntity"],
    join_type="left anti"
)

assert missing_in_gold.num_rows == 0
assert missing_in_validation.num_rows == 0


# Flatten validation context posts
validation_contexts = llm_validation["Context"].combine_chunks()

validation_contents = pc.list_flatten(validation_contexts)
parent_indices = pc.list_parent_indices(validation_contexts)

validation_context_posts = pa.table({
    "UserId": pc.take(
        llm_validation["UserId"].combine_chunks(),
        parent_indices
    ),
    "TargetEntity": pc.take(
        llm_validation["TargetEntity"].combine_chunks(),
        parent_indices
    ),
    "Content": validation_contents
})

validation_context_posts = validation_context_posts.append_column(
    "ContextPostId",
    pa.array(
        np.arange(validation_context_posts.num_rows),
        type=pa.int64()
    )
)


# Match through UserId + Content
validation_matches = validation_context_posts.join(
    history_lookup,
    keys=["UserId", "Content"],
    join_type="left outer"
)

num_unmatched = validation_matches["PostId"].null_count

print(f"Human gold users: {pc.count_distinct(human_gold['UserId']).as_py():,}")
print(f"Human gold user-target rows: {human_gold.num_rows:,}")
print(f"Validation context posts: {validation_context_posts.num_rows:,}")
print(f"Unmatched context posts: {num_unmatched:,}")

assert num_unmatched == 0, (
    "Some validation context posts could not be matched "
    "to the original posting history."
)


# Determine whether each context post has one or multiple possible PostIds
validation_match_summary = (
    validation_matches
    .group_by([
        "ContextPostId",
        "UserId"
    ])
    .aggregate([
        ("PostId", "count_distinct"),
        ("PostId", "min")
    ])
)

validation_match_summary = validation_match_summary.rename_columns([
    "ContextPostId",
    "UserId",
    "NumMatches",
    "MatchedPostId"
])

resolved_validation_posts = validation_context_posts.join(
    validation_match_summary,
    keys=["ContextPostId", "UserId"],
    join_type="left outer"
)

is_ambiguous = pc.greater(
    resolved_validation_posts["NumMatches"],
    1
)

resolved_validation_posts = resolved_validation_posts.append_column(
    "PostId",
    pc.if_else(
        is_ambiguous,
        pa.scalar(
            None,
            type=resolved_validation_posts["MatchedPostId"].type
        ),
        resolved_validation_posts["MatchedPostId"]
    )
)

resolved_validation_posts = resolved_validation_posts.append_column(
    "IsAmbiguous",
    is_ambiguous
)


# Recover PostTime using UserId + PostId
post_time_lookup = history_lookup.select([
    "UserId",
    "PostId",
    "PostTime"
])

resolved_validation_posts = resolved_validation_posts.join(
    post_time_lookup,
    keys=["UserId", "PostId"],
    join_type="left outer"
)

assert (
    resolved_validation_posts.num_rows
    == validation_context_posts.num_rows
), "Matching changed the number of validation context posts."

resolved_validation_posts = resolved_validation_posts.select([
    "ContextPostId",
    "UserId",
    "TargetEntity",
    "PostId",
    "Content",
    "PostTime",
    "IsAmbiguous"
])

display(resolved_validation_posts.slice(0, 10).to_pandas())

Human gold users: 445
Human gold user-target rows: 890
Validation context posts: 6,829
Unmatched context posts: 0


,ContextPostId,UserId,TargetEntity,PostId,Content,PostTime,IsAmbiguous
0,0,265,Harris,17388,"""Makary is a close ally of Robert F. Kennedy J...",2024-11-23T01:23:37.373Z,False
1,1,265,Harris,17391,"""In recent months, Dr. Makary has publicly sou...",2024-11-23T01:55:18.522Z,False
2,2,265,Harris,15631967,Looking forward to an important discussion thi...,2023-10-22T06:04:48.353Z,False
3,3,265,Harris,17418,“You really can’t engage with someone like tha...,2024-11-27T22:21:32.328Z,False
4,4,265,Harris,17392,"""Makary gained prominence on Fox News and othe...",2024-11-23T02:13:12.122Z,False
5,5,265,Harris,2397753,I just set up a starter pack for bioethics/app...,2024-11-24T18:56:45.167Z,False
6,6,265,Harris,17394,"""By and large, the nominees don’t have experie...",2024-11-23T18:13:55.398Z,False
7,7,265,Harris,9572174,Unsettling convo with @anonymous on whistleblo...,2024-11-21T06:54:47.276Z,False
8,8,265,Trump,17391,"""In recent months, Dr. Makary has publicly sou...",2024-11-23T01:55:18.522Z,False
9,9,265,Trump,11337690,"NEWS: Celebrity doctor Mehmet Oz, whom Trump w...",2024-11-22T19:02:07.771Z,False


In [21]:
# Apply the same chronological ordering as for the LLM data
resolved_validation_posts = resolved_validation_posts.sort_by([
    ("UserId", "ascending"),
    ("TargetEntity", "ascending"),
    ("PostTime", "descending"),
    ("ContextPostId", "ascending")
])

grouped_validation_context = defaultdict(list)

for row in resolved_validation_posts.to_pylist():
    key = (row["UserId"], row["TargetEntity"])

    grouped_validation_context[key].append({
        "PostId": row["PostId"],
        "Content": row["Content"],
        "PostTime": row["PostTime"],
        "IsAmbiguous": row["IsAmbiguous"]
    })


# Build contexts in the same order as the human gold rows
context_lists = []

for row in human_gold.select([
    "UserId",
    "TargetEntity"
]).to_pylist():

    key = (row["UserId"], row["TargetEntity"])
    context_lists.append(grouped_validation_context[key])


num_context_posts = [
    len(posts)
    for posts in context_lists
]

num_ambiguous_timestamps = [
    sum(post["IsAmbiguous"] for post in posts)
    for posts in context_lists
]


human_context_data = pa.table({
    "UserId": human_gold["UserId"],
    "TargetEntity": human_gold["TargetEntity"],
    "StanceLabel": human_gold["StanceLabel"],
    "ContextPosts": pa.array(
        context_lists,
        type=context_type
    ),
    "NumContextPosts": pa.array(
        num_context_posts,
        type=pa.int16()
    ),
    "NumAmbiguousTimestamps": pa.array(
        num_ambiguous_timestamps,
        type=pa.int16()
    )
})


# Final structural checks
assert human_context_data.num_rows == human_gold.num_rows

assert sum(num_context_posts) == resolved_validation_posts.num_rows

assert max(num_context_posts) <= 10


rows_with_ambiguity = sum(
    count > 0
    for count in num_ambiguous_timestamps
)

print(f"Human test rows: {human_context_data.num_rows:,}")
print(f"Total context posts: {sum(num_context_posts):,}")
print(
    f"Rows with at least one ambiguous timestamp: "
    f"{rows_with_ambiguity:,} "
    f"({rows_with_ambiguity / human_context_data.num_rows:.2%})"
)

display(human_context_data.slice(0, 5).to_pandas())


human_context_path = (
    f"{preprocessed_path}/human_user_target_contexts.parquet"
)

pq.write_table(
    human_context_data,
    human_context_path,
    compression="snappy"
)

print(f"\nSaved to: {human_context_path}")

Human test rows: 890
Total context posts: 6,829
Rows with at least one ambiguous timestamp: 44 (4.94%)


,UserId,TargetEntity,StanceLabel,ContextPosts,NumContextPosts,NumAmbiguousTimestamps
0,186791,Trump,Against,"[{'PostId': 2625248, 'Content': 'Lmao #fucktru...",8,0
1,88089,Trump,Against,"[{'PostId': 11730207, 'Content': '""I can't get...",9,0
2,254114,Trump,Against,"[{'PostId': 11610857, 'Content': 'This has alw...",7,0
3,77504,Trump,Against,"[{'PostId': 12124994, 'Content': 'Tomorrow mor...",9,0
4,132412,Trump,Against,"[{'PostId': 2754321, 'Content': 'We need prote...",8,0



Saved to: ../data/preprocessed/human_user_target_contexts.parquet


In [22]:
# Quick sanity check to ensure successful ordering for both llm and human data
def check_context_order(table, name):
    for posts in table["ContextPosts"].to_pylist():

        timestamps = [
            post["PostTime"]
            for post in posts
            if not post["IsAmbiguous"]
        ]

        # Non-ambiguous posts must be newest -> oldest
        assert timestamps == sorted(timestamps, reverse=True), (
            f"{name}: ContextPosts are not ordered correctly."
        )

        # Once an ambiguous post appears, all following posts must be ambiguous
        seen_ambiguous = False

        for post in posts:
            if post["IsAmbiguous"]:
                seen_ambiguous = True
                assert post["PostTime"] is None
            else:
                assert not seen_ambiguous, (
                    f"{name}: Timestamped post appears after an ambiguous post."
                )

    print(f"{name}: context ordering correct.")


check_context_order(llm_context_data, "LLM data")
check_context_order(human_context_data, "Human data")

LLM data: context ordering correct.
Human data: context ordering correct.


---
## 7. Create the final train and validation datasets

The human-annotated users are kept completely separate as the final test set. The remaining LLM-labelled users are split into training and validation data using an 80/20 split.

The split is performed at the **user level** rather than at the user-target level. This ensures that both target-specific rows of the same user always belong to the same split and prevents information leakage.

After creating the split, we retain only the information required for the downstream modelling experiments:

- `UserId`
- `TargetEntity`
- `StanceLabel`
- `ContextPosts`

Within `ContextPosts`, we retain:

- `Content`, which is the actual model input
- `PostTime`, which is required for temporal k-selection
- `IsAmbiguous`, which identifies posts that cannot be reliably ordered by time

`PostId`, LLM confidence scores, and the context-count summary columns are not required for modelling and are therefore removed from the final datasets. They remain available in the richer intermediate preprocessing tables.

In [23]:
from sklearn.model_selection import train_test_split


llm_users = np.array(
    pc.unique(llm_context_data["UserId"]).to_pylist()
)

# Split LLM-labelled users at the user level
train_users, validation_users = train_test_split(
    llm_users,
    test_size=0.2,
    random_state=42
)

train_mask = pc.is_in(
    llm_context_data["UserId"],
    value_set=pa.array(
        train_users,
        type=llm_context_data["UserId"].type
    )
)

validation_mask = pc.is_in(
    llm_context_data["UserId"],
    value_set=pa.array(
        validation_users,
        type=llm_context_data["UserId"].type
    )
)

train_rich = llm_context_data.filter(train_mask)
validation_rich = llm_context_data.filter(validation_mask)


# Verify that all three user sets are disjoint
train_ids = set(
    pc.unique(train_rich["UserId"]).to_pylist()
)

validation_ids = set(
    pc.unique(validation_rich["UserId"]).to_pylist()
)

test_ids = set(
    pc.unique(human_context_data["UserId"]).to_pylist()
)

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

assert (
    train_rich.num_rows + validation_rich.num_rows
    == llm_context_data.num_rows
)

print(f"Train users: {len(train_ids):,}")
print(f"Validation users: {len(validation_ids):,}")
print(f"Human test users: {len(test_ids):,}")

print(f"\nTrain rows: {train_rich.num_rows:,}")
print(f"Validation rows: {validation_rich.num_rows:,}")
print(f"Human test rows: {human_context_data.num_rows:,}")

print("\nNo user overlap between train, validation, and test.")

Train users: 6,417
Validation users: 1,605
Human test users: 445

Train rows: 12,834
Validation rows: 3,210
Human test rows: 890

No user overlap between train, validation, and test.


In [24]:
# Context structure required for downstream experiments
rich_context_type = llm_context_data.schema.field(
    "ContextPosts"
).type.value_type

model_context_type = pa.list_(
    pa.struct([
        pa.field(
            "Content",
            rich_context_type.field("Content").type
        ),
        pa.field(
            "PostTime",
            rich_context_type.field("PostTime").type
        ),
        pa.field(
            "IsAmbiguous",
            pa.bool_()
        )
    ])
)


def prepare_model_data(table):
    context_lists = []

    for posts in table["ContextPosts"].to_pylist():
        context_lists.append([
            {
                "Content": post["Content"],
                "PostTime": post["PostTime"],
                "IsAmbiguous": post["IsAmbiguous"]
            }
            for post in posts
        ])

    return pa.table({
        "UserId": table["UserId"],
        "TargetEntity": table["TargetEntity"],
        "StanceLabel": table["StanceLabel"],
        "ContextPosts": pa.array(
            context_lists,
            type=model_context_type
        )
    })


train_data = prepare_model_data(train_rich)
validation_data = prepare_model_data(validation_rich)
human_test_data = prepare_model_data(human_context_data)


# Final structure should be identical across all three datasets
expected_columns = [
    "UserId",
    "TargetEntity",
    "StanceLabel",
    "ContextPosts"
]

assert train_data.column_names == expected_columns
assert validation_data.column_names == expected_columns
assert human_test_data.column_names == expected_columns

print("Final columns:")
print(train_data.column_names)

print("\nContextPosts type:")
print(train_data["ContextPosts"].type)

print("\nFinal row counts:")
print(f"Train: {train_data.num_rows:,}")
print(f"Validation: {validation_data.num_rows:,}")
print(f"Human test: {human_test_data.num_rows:,}")

Final columns:
['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']

ContextPosts type:
list<item: struct<Content: string, PostTime: string, IsAmbiguous: bool>>

Final row counts:
Train: 12,834
Validation: 3,210
Human test: 890


---
## 8. Save the final model datasets

The preprocessing pipeline is now complete.

We save the three model-ready datasets:

- `train.parquet`: LLM-labelled training data
- `validation.parquet`: LLM-labelled validation data
- `human_test.parquet`: held-out human gold test data

All three datasets contain the same structure:

- `UserId`
- `TargetEntity`
- `StanceLabel`
- `ContextPosts`

Each context post contains only `Content`, `PostTime`, and `IsAmbiguous`. The richer intermediate tables are kept separately for debugging and reproducibility.

In [25]:
train_path = f"{preprocessed_path}/train.parquet"
validation_path = f"{preprocessed_path}/validation.parquet"
human_test_path = f"{preprocessed_path}/human_test.parquet"


pq.write_table(
    train_data,
    train_path,
    compression="snappy"
)

pq.write_table(
    validation_data,
    validation_path,
    compression="snappy"
)

pq.write_table(
    human_test_data,
    human_test_path,
    compression="snappy"
)

print(f"Saved train set to: {train_path}")
print(f"Saved validation set to: {validation_path}")
print(f"Saved human test set to: {human_test_path}")

Saved train set to: ../data/preprocessed/train.parquet
Saved validation set to: ../data/preprocessed/validation.parquet
Saved human test set to: ../data/preprocessed/human_test.parquet
